# 02 - Classify Dependabot Usage

This notebook joins the raw repo-level Dependabot evidence from `01_mine_dependabot_evidence.ipynb` back to every RQ2 CVE-UP-DOWN adoption pair.

It produces deterministic repo-level and pair-level classifications without making GitHub API calls.

## Inputs and Outputs

Inputs:

- `../../data/rq2/rq2_master_pairwise.csv`
- `data/dependabot_repo_evidence_raw.csv`

Outputs:

- `data/dependabot_repo_classified.csv`
- `data/dependabot_pairwise_classified.csv`
- `data/dependabot_repo_usage_summary.csv`
- `data/dependabot_pairwise_usage_summary.csv`

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parents[1]
INPUT_CSV = ROOT / 'data' / 'rq2' / 'rq2_master_pairwise.csv'
DATA_DIR = ROOT / 'data' / 'rq3'
RAW_EVIDENCE_CSV = DATA_DIR / 'dependabot_repo_evidence_raw.csv'

REPO_CLASSIFIED_CSV = DATA_DIR / 'dependabot_repo_classified.csv'
PAIRWISE_CLASSIFIED_CSV = DATA_DIR / 'dependabot_pairwise_classified.csv'
REPO_SUMMARY_CSV = DATA_DIR / 'dependabot_repo_usage_summary.csv'
REPO_USAGE_TYPE_SUMMARY_CSV = DATA_DIR / 'dependabot_repo_usage_type_summary.csv'
PAIRWISE_SUMMARY_CSV = DATA_DIR / 'dependabot_pairwise_usage_summary.csv'

DATA_DIR.mkdir(parents=True, exist_ok=True)

MIN_DEPENDABOT_EVIDENCE_DATE = pd.Timestamp('2017-01-01T00:00:00Z')

## Load Data

In [2]:
pairs = pd.read_csv(INPUT_CSV)
evidence_raw = pd.read_csv(RAW_EVIDENCE_CSV)

required_pair_cols = {'CVE', 'upstream_GA', 'downstream_GA', 'downstream_repo', 'commit_url', 'adoption_date'}
required_evidence_cols = {
    'repo', 'mined_at_utc', 'mining_status',
    'has_dependabot_pr', 'first_dependabot_pr_date', 'first_dependabot_pr_url',
    'has_dependabot_config', 'dependabot_config_path', 'dependabot_config_created_date',
}

missing_pair_cols = required_pair_cols - set(pairs.columns)
missing_evidence_cols = required_evidence_cols - set(evidence_raw.columns)
if missing_pair_cols:
    raise ValueError(f'Missing pairwise columns: {sorted(missing_pair_cols)}')
if missing_evidence_cols:
    raise ValueError(f'Missing evidence columns: {sorted(missing_evidence_cols)}')

print(f'Pairwise rows: {len(pairs):,}')
print(f'Raw evidence rows: {len(evidence_raw):,}')
display(pairs.head(3))
display(evidence_raw.head(3))

Pairwise rows: 1,677
Raw evidence rows: 742


,CVE,upstream_GA,downstream_GA,downstream_repo,commit_url,adoption_date,commit_n_CVEs,class,pattern,original_pattern,...,delay_fix,delay_release,delay_disclosure,mitigation_stage,disclosure_year,dependents,downstream_usage_num,downstream_loc,downstream_class_num,upstream_stars_at_fix
0,CVE-2021-37714,org.jsoup:jsoup,com.github.btheu.estivate:estivate,btheu/estivate,https://github.com/btheu/estivate/commit/22adf...,2021-12-03 15:44:15,1,Transparent,T1,T1,...,110.134062,110.003472,107.655729,After Disclosure,2021,244,0,3801.0,102.0,8521.0
1,CVE-2021-37714,org.jsoup:jsoup,com.jcabi:jcabi-http,jcabi/jcabi-http,https://github.com/jcabi/jcabi-http/commit/1a2...,2021-08-16 23:00:43,1,Transparent,T1,T1,...,1.437164,1.306574,-1.041169,Release->Disclosure,2021,244,0,3957.5,89.5,8521.0
2,CVE-2021-37714,org.jsoup:jsoup,in.ashwanthkumar:gocd-java-client,ashwanthkumar/gocd-java-client,https://github.com/ashwanthkumar/gocd-java-cli...,2021-09-30 23:03:50,1,Transparent,T1,T1,...,46.439329,46.308738,43.960995,After Disclosure,2021,244,0,1339.5,27.0,8521.0


,repo,repo_url,mined_at_utc,mining_status,error_type,error_message,has_dependabot_pr,first_dependabot_pr_date,first_dependabot_pr_url,first_dependabot_pr_number,first_dependabot_pr_author,first_dependabot_pr_title,first_dependabot_pr_query_author,dependabot_pr_search_queries,has_dependabot_config,dependabot_config_path,dependabot_config_created_date,dependabot_config_created_sha,dependabot_config_created_url,dependabot_config_created_message
0,42BV/jarb,https://github.com/42BV/jarb,2026-08-14T17:00:36.520962+00:00,ok,NaN,NaN,True,2020-01-21T21:11:51Z,https://github.com/42BV/jarb/pull/53,53.0,dependabot,Bump spring.version from 5.1.3.RELEASE to 5.2....,app/dependabot,repo:42BV/jarb is:pr author:app/dependabot cre...,False,NaN,NaN,NaN,NaN,NaN
1,52North/arctic-sea,https://github.com/52North/arctic-sea,2026-08-14T17:00:42.593324+00:00,ok,NaN,NaN,True,2019-08-01T21:59:32Z,https://github.com/52North/arctic-sea/pull/47,47.0,dependabot,Bump version.jackson from 2.9.9 to 2.10.0.pr1,app/dependabot,repo:52North/arctic-sea is:pr author:app/depen...,True,.github/dependabot.yml,2021-04-29T15:25:46Z,fe7416a0a9a56c960616fe58de41f1e5b0e19049,https://github.com/52North/arctic-sea/commit/f...,Upgrade to GitHub-native Dependabot
2,52North/series-hibernate,https://github.com/52North/series-hibernate,2026-08-14T17:00:50.205286+00:00,ok,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,repo:52North/series-hibernate is:pr author:app...,True,.github/dependabot.yml,2021-04-29T15:25:57Z,924fab33f054d8521a2e2ade95609ec9f0aba51b,https://github.com/52North/sensorweb-server-db...,Upgrade to GitHub-native Dependabot


## Normalize and Deduplicate Evidence

In [3]:
def normalize_repo(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    if not text:
        return None

    text = text.replace('git@github.com:', 'https://github.com/')
    text = text.replace('http://github.com/', 'https://github.com/')
    if text.startswith('https://github.com/'):
        text = text[len('https://github.com/'):]
    if text.startswith('github.com/'):
        text = text[len('github.com/'):]
    text = text.split('#', 1)[0].split('?', 1)[0].strip('/')
    if text.endswith('.git'):
        text = text[:-4]

    parts = [p for p in text.split('/') if p]
    if len(parts) < 2:
        return None
    return f'{parts[0]}/{parts[1]}'

def parse_bool(series):
    return series.astype(str).str.strip().str.lower().map({
        'true': True,
        '1': True,
        'yes': True,
        'false': False,
        '0': False,
        'no': False,
        'nan': False,
        'none': False,
        '': False,
    }).fillna(False)

pairs = pairs.copy()
pairs['normalized_downstream_repo'] = pairs['downstream_repo'].map(normalize_repo)
pairs['adoption_date_utc'] = pd.to_datetime(pairs['adoption_date'], errors='coerce', utc=True)

evidence = evidence_raw.copy()
evidence['repo'] = evidence['repo'].map(normalize_repo)
evidence['mined_at_utc_parsed'] = pd.to_datetime(evidence['mined_at_utc'], errors='coerce', utc=True)
evidence['has_dependabot_pr'] = parse_bool(evidence['has_dependabot_pr'])
evidence['has_dependabot_config'] = parse_bool(evidence['has_dependabot_config'])
evidence['first_dependabot_pr_date_utc'] = pd.to_datetime(evidence['first_dependabot_pr_date'], errors='coerce', utc=True)
evidence['dependabot_config_created_date_utc'] = pd.to_datetime(evidence['dependabot_config_created_date'], errors='coerce', utc=True)

# Keep the latest successful row for each repo. This makes reruns/appends safe.
evidence_ok = evidence[evidence['mining_status'].eq('ok')].copy()
evidence_ok = evidence_ok.sort_values(['repo', 'mined_at_utc_parsed']).drop_duplicates('repo', keep='last')

pair_repo_count = pairs['normalized_downstream_repo'].dropna().nunique()
evidence_repo_count = evidence_ok['repo'].dropna().nunique()
missing_evidence_repos = sorted(set(pairs['normalized_downstream_repo'].dropna()) - set(evidence_ok['repo'].dropna()))

print(f'Unique downstream repos in pairs: {pair_repo_count:,}')
print(f'Unique successful evidence repos: {evidence_repo_count:,}')
print(f'Repos missing successful evidence: {len(missing_evidence_repos):,}')
if missing_evidence_repos:
    print(missing_evidence_repos[:20])

Unique downstream repos in pairs: 742
Unique successful evidence repos: 742
Repos missing successful evidence: 0


## Repo-Level Classification

In [4]:
repo_classified = evidence_ok.copy()

repo_classified['has_dependabot_pr'] = repo_classified['has_dependabot_pr'] & repo_classified['first_dependabot_pr_date_utc'].notna()
repo_classified['has_dependabot_config'] = repo_classified['has_dependabot_config'] & repo_classified['dependabot_config_created_date_utc'].notna()
repo_classified['uses_dependabot_ever'] = repo_classified['has_dependabot_pr'] | repo_classified['has_dependabot_config']

same_day = (
    repo_classified['has_dependabot_pr']
    & repo_classified['has_dependabot_config']
    & (repo_classified['first_dependabot_pr_date_utc'].dt.date == repo_classified['dependabot_config_created_date_utc'].dt.date)
)
security_then_version = (
    repo_classified['has_dependabot_pr']
    & repo_classified['has_dependabot_config']
    & (repo_classified['first_dependabot_pr_date_utc'] < repo_classified['dependabot_config_created_date_utc'])
    & ~same_day
)
version_then_security = (
    repo_classified['has_dependabot_pr']
    & repo_classified['has_dependabot_config']
    & (repo_classified['dependabot_config_created_date_utc'] < repo_classified['first_dependabot_pr_date_utc'])
    & ~same_day
)

repo_classified['dependabot_lifecycle'] = np.select(
    [
        ~repo_classified['uses_dependabot_ever'],
        repo_classified['has_dependabot_pr'] & ~repo_classified['has_dependabot_config'],
        ~repo_classified['has_dependabot_pr'] & repo_classified['has_dependabot_config'],
        same_day,
        security_then_version,
        version_then_security,
    ],
    [
        'never_detected',
        'security_only_ever',
        'version_config_only_ever',
        'same_day_security_and_version',
        'security_then_version',
        'version_then_security',
    ],
    default='unclassified',
)

repo_classified['dependabot_usage_type_ever'] = np.select(
    [
        ~repo_classified['uses_dependabot_ever'],
        repo_classified['has_dependabot_pr'] & ~repo_classified['has_dependabot_config'],
        ~repo_classified['has_dependabot_pr'] & repo_classified['has_dependabot_config'],
        repo_classified['has_dependabot_pr'] & repo_classified['has_dependabot_config'],
    ],
    [
        'not_using_dependabot',
        'security_updates_only',
        'version_updates_config_only',
        'both_security_and_version_updates',
    ],
    default='unclassified',
)
repo_classified['security_first_version_later_ever'] = repo_classified['dependabot_lifecycle'].eq('security_then_version')

repo_classified['days_security_to_version'] = (
    repo_classified['dependabot_config_created_date_utc'] - repo_classified['first_dependabot_pr_date_utc']
).dt.total_seconds() / 86400
repo_classified.loc[~security_then_version, 'days_security_to_version'] = np.nan

repo_output_cols = [
    'repo', 'repo_url', 'uses_dependabot_ever', 'dependabot_usage_type_ever',
    'dependabot_lifecycle', 'security_first_version_later_ever',
    'has_dependabot_pr', 'first_dependabot_pr_date', 'first_dependabot_pr_url',
    'first_dependabot_pr_number', 'first_dependabot_pr_author', 'first_dependabot_pr_title',
    'has_dependabot_config', 'dependabot_config_path', 'dependabot_config_created_date',
    'dependabot_config_created_sha', 'dependabot_config_created_url',
    'days_security_to_version', 'mined_at_utc',
]
repo_classified[repo_output_cols].to_csv(REPO_CLASSIFIED_CSV, index=False)

print(f'Wrote {REPO_CLASSIFIED_CSV} with {len(repo_classified):,} rows')
display(repo_classified[repo_output_cols].head())

Wrote ../../data/rq3/dependabot_repo_classified.csv with 742 rows


,repo,repo_url,uses_dependabot_ever,dependabot_usage_type_ever,dependabot_lifecycle,security_first_version_later_ever,has_dependabot_pr,first_dependabot_pr_date,first_dependabot_pr_url,first_dependabot_pr_number,first_dependabot_pr_author,first_dependabot_pr_title,has_dependabot_config,dependabot_config_path,dependabot_config_created_date,dependabot_config_created_sha,dependabot_config_created_url,days_security_to_version,mined_at_utc
0,42BV/jarb,https://github.com/42BV/jarb,True,security_updates_only,security_only_ever,False,True,2020-01-21T21:11:51Z,https://github.com/42BV/jarb/pull/53,53.0,dependabot,Bump spring.version from 5.1.3.RELEASE to 5.2....,False,NaN,NaN,NaN,NaN,NaN,2026-08-14T17:00:36.520962+00:00
1,52North/arctic-sea,https://github.com/52North/arctic-sea,True,both_security_and_version_updates,security_then_version,True,True,2019-08-01T21:59:32Z,https://github.com/52North/arctic-sea/pull/47,47.0,dependabot,Bump version.jackson from 2.9.9 to 2.10.0.pr1,True,.github/dependabot.yml,2021-04-29T15:25:46Z,fe7416a0a9a56c960616fe58de41f1e5b0e19049,https://github.com/52North/arctic-sea/commit/f...,636.726551,2026-08-14T17:00:42.593324+00:00
2,52North/series-hibernate,https://github.com/52North/series-hibernate,True,version_updates_config_only,version_config_only_ever,False,False,NaN,NaN,NaN,NaN,NaN,True,.github/dependabot.yml,2021-04-29T15:25:57Z,924fab33f054d8521a2e2ade95609ec9f0aba51b,https://github.com/52North/sensorweb-server-db...,NaN,2026-08-14T17:00:50.205286+00:00
3,6tail/nlf2-maven,https://github.com/6tail/nlf2-maven,True,security_updates_only,security_only_ever,False,True,2019-10-29T21:06:00Z,https://github.com/6tail/nlf2-maven/pull/1,1.0,dependabot,Bump c3p0 from 0.9.5.2 to 0.9.5.4 in /nlf2-plu...,False,NaN,NaN,NaN,NaN,NaN,2026-08-14T17:00:59.268097+00:00
4,88250/latke,https://github.com/88250/latke,True,security_updates_only,security_only_ever,False,True,2021-02-08T21:25:30Z,https://github.com/88250/latke/pull/37,37.0,dependabot,⬆️ Bump netty.version from 4.1.49.Final to 4.1...,False,NaN,NaN,NaN,NaN,NaN,2026-08-14T17:01:06.101623+00:00


## Pair-Level Adoption-Time Classification

In [5]:
evidence_join_cols = [
    'repo', 'uses_dependabot_ever', 'dependabot_usage_type_ever', 'dependabot_lifecycle',
    'has_dependabot_pr', 'first_dependabot_pr_date', 'first_dependabot_pr_date_utc',
    'first_dependabot_pr_url', 'first_dependabot_pr_number', 'first_dependabot_pr_author',
    'first_dependabot_pr_title', 'has_dependabot_config', 'dependabot_config_path',
    'dependabot_config_created_date', 'dependabot_config_created_date_utc',
    'dependabot_config_created_sha', 'dependabot_config_created_url', 'days_security_to_version',
]

pairwise = pairs.merge(
    repo_classified[evidence_join_cols],
    how='left',
    left_on='normalized_downstream_repo',
    right_on='repo',
)

pairwise['has_successful_dependabot_mining'] = pairwise['repo'].notna()
pairwise['security_evidence_at_adoption'] = (
    pairwise['has_dependabot_pr'].fillna(False)
    & pairwise['first_dependabot_pr_date_utc'].notna()
    & pairwise['adoption_date_utc'].notna()
    & (pairwise['first_dependabot_pr_date_utc'] <= pairwise['adoption_date_utc'])
)
pairwise['version_evidence_at_adoption'] = (
    pairwise['has_dependabot_config'].fillna(False)
    & pairwise['dependabot_config_created_date_utc'].notna()
    & pairwise['adoption_date_utc'].notna()
    & (pairwise['dependabot_config_created_date_utc'] <= pairwise['adoption_date_utc'])
)

pairwise['dependabot_status_at_adoption'] = np.select(
    [
        pairwise['version_evidence_at_adoption'],
        pairwise['security_evidence_at_adoption'] & ~pairwise['version_evidence_at_adoption'],
        ~pairwise['security_evidence_at_adoption'] & ~pairwise['version_evidence_at_adoption'],
    ],
    [
        'version_updates_enabled_at_adoption',
        'security_only_at_adoption',
        'no_dependabot_evidence_at_adoption',
    ],
    default='unclassified',
)

pairwise['uses_dependabot_at_adoption'] = pairwise['dependabot_status_at_adoption'].ne('no_dependabot_evidence_at_adoption')
pairwise['any_dependabot_at_adoption'] = np.where(
    pairwise['uses_dependabot_at_adoption'],
    'any_dependabot_at_adoption',
    'no_dependabot_at_adoption',
)
pairwise['dependabot_setting_at_adoption'] = pairwise['dependabot_status_at_adoption'].map({
    'no_dependabot_evidence_at_adoption': 'No Dependabot evidence',
    'security_only_at_adoption': 'Security updates only',
    'version_updates_enabled_at_adoption': 'Version updates enabled',
}).fillna('Unclassified')
pairwise['adoption_time_setting_precise'] = pairwise['dependabot_status_at_adoption'].map({
    'no_dependabot_evidence_at_adoption': 'No Dependabot evidence',
    'security_only_at_adoption': 'Security updates only',
    'version_updates_enabled_at_adoption': 'Version updates enabled',
}).fillna('Unclassified')
pairwise['include_in_precise_setting_delay_analysis'] = pairwise['dependabot_status_at_adoption'].isin([
    'no_dependabot_evidence_at_adoption',
    'security_only_at_adoption',
    'version_updates_enabled_at_adoption',
])

pairwise['security_then_version_later_ever'] = pairwise['dependabot_lifecycle'].eq('security_then_version')
pairwise['repo_strategy_security_then_version'] = pairwise['dependabot_lifecycle'].eq('security_then_version')
pairwise['security_only_at_adoption_then_version_later'] = (
    pairwise['dependabot_status_at_adoption'].eq('security_only_at_adoption')
    & pairwise['dependabot_config_created_date_utc'].notna()
    & pairwise['adoption_date_utc'].notna()
    & (pairwise['dependabot_config_created_date_utc'] > pairwise['adoption_date_utc'])
)

pairwise['days_pr_before_adoption'] = (
    pairwise['adoption_date_utc'] - pairwise['first_dependabot_pr_date_utc']
).dt.total_seconds() / 86400
pairwise.loc[~pairwise['security_evidence_at_adoption'], 'days_pr_before_adoption'] = np.nan

pairwise['days_config_before_adoption'] = (
    pairwise['adoption_date_utc'] - pairwise['dependabot_config_created_date_utc']
).dt.total_seconds() / 86400
pairwise.loc[~pairwise['version_evidence_at_adoption'], 'days_config_before_adoption'] = np.nan

pairwise = pairwise.drop(columns=['repo'])
pairwise.to_csv(PAIRWISE_CLASSIFIED_CSV, index=False)

print(f'Wrote {PAIRWISE_CLASSIFIED_CSV} with {len(pairwise):,} rows')
display(pairwise[['CVE', 'upstream_GA', 'downstream_GA', 'normalized_downstream_repo', 'adoption_date', 'dependabot_status_at_adoption', 'adoption_time_setting_precise', 'include_in_precise_setting_delay_analysis', 'any_dependabot_at_adoption', 'dependabot_lifecycle']].head())

Wrote ../../data/rq3/dependabot_pairwise_classified.csv with 1,677 rows


,CVE,upstream_GA,downstream_GA,normalized_downstream_repo,adoption_date,dependabot_status_at_adoption,adoption_time_setting_precise,include_in_precise_setting_delay_analysis,any_dependabot_at_adoption,dependabot_lifecycle
0,CVE-2021-37714,org.jsoup:jsoup,com.github.btheu.estivate:estivate,btheu/estivate,2021-12-03 15:44:15,security_only_at_adoption,Security updates only,True,any_dependabot_at_adoption,security_only_ever
1,CVE-2021-37714,org.jsoup:jsoup,com.jcabi:jcabi-http,jcabi/jcabi-http,2021-08-16 23:00:43,version_updates_enabled_at_adoption,Version updates enabled,True,any_dependabot_at_adoption,security_then_version
2,CVE-2021-37714,org.jsoup:jsoup,in.ashwanthkumar:gocd-java-client,ashwanthkumar/gocd-java-client,2021-09-30 23:03:50,version_updates_enabled_at_adoption,Version updates enabled,True,any_dependabot_at_adoption,security_then_version
3,CVE-2021-37714,org.jsoup:jsoup,tech.grasshopper:pdfextentreporter,grasshopper7/pdfextentreporter,2022-03-01 12:31:07,no_dependabot_evidence_at_adoption,No Dependabot evidence,True,no_dependabot_at_adoption,never_detected
4,CVE-2021-37714,org.jsoup:jsoup,de.trustable.ca3s.core:ca-3-s,kuehne-trustable-de/ca3sCore,2021-08-23 21:32:03,security_only_at_adoption,Security updates only,True,any_dependabot_at_adoption,security_only_ever


## Summary Counts

In [6]:
repo_summary = repo_classified.groupby('dependabot_lifecycle', dropna=False).agg(
    downstream_projects=('repo', 'nunique')
).reset_index().sort_values('downstream_projects', ascending=False)
repo_summary['percent_of_downstream_projects'] = repo_summary['downstream_projects'] / repo_classified['repo'].nunique() * 100

repo_usage_type_summary = repo_classified.groupby('dependabot_usage_type_ever', dropna=False).agg(
    downstream_projects=('repo', 'nunique')
).reset_index().sort_values('downstream_projects', ascending=False)
repo_usage_type_summary['percent_of_downstream_projects'] = repo_usage_type_summary['downstream_projects'] / repo_classified['repo'].nunique() * 100

repo_use_summary = pd.DataFrame([
    {'metric': 'unique_downstream_projects', 'count': repo_classified['repo'].nunique()},
    {'metric': 'uses_dependabot_ever', 'count': int(repo_classified['uses_dependabot_ever'].sum())},
    {'metric': 'no_dependabot_evidence_ever', 'count': int((~repo_classified['uses_dependabot_ever']).sum())},
    {'metric': 'has_dependabot_pr', 'count': int(repo_classified['has_dependabot_pr'].sum())},
    {'metric': 'has_dependabot_config', 'count': int(repo_classified['has_dependabot_config'].sum())},
    {'metric': 'has_both_pr_and_config', 'count': int((repo_classified['has_dependabot_pr'] & repo_classified['has_dependabot_config']).sum())},
    {'metric': 'pr_only', 'count': int((repo_classified['has_dependabot_pr'] & ~repo_classified['has_dependabot_config']).sum())},
    {'metric': 'config_only', 'count': int((~repo_classified['has_dependabot_pr'] & repo_classified['has_dependabot_config']).sum())},
    {'metric': 'security_then_version', 'count': int(repo_classified['dependabot_lifecycle'].eq('security_then_version').sum())},
])
repo_use_summary['percent_of_downstream_projects'] = repo_use_summary['count'] / repo_classified['repo'].nunique() * 100

pairwise_summary = pairwise.groupby('dependabot_status_at_adoption', dropna=False).agg(
    pairwise_rows=('dependabot_status_at_adoption', 'size'),
    downstream_projects=('normalized_downstream_repo', 'nunique'),
    cves=('CVE', 'nunique'),
).reset_index().sort_values('pairwise_rows', ascending=False)
pairwise_summary['percent_of_pairwise_rows'] = pairwise_summary['pairwise_rows'] / len(pairwise) * 100

repo_use_summary.to_csv(REPO_SUMMARY_CSV, index=False)
repo_usage_type_summary.to_csv(REPO_USAGE_TYPE_SUMMARY_CSV, index=False)
pairwise_summary.to_csv(PAIRWISE_SUMMARY_CSV, index=False)

print('Repo-level use summary')
display(repo_use_summary)

print('Repo-level lifecycle summary')
display(repo_summary)

print('Repo-level usage type summary')
display(repo_usage_type_summary)

print('Pair-level adoption-time summary')
display(pairwise_summary)

Repo-level use summary


,metric,count,percent_of_downstream_projects
0,unique_downstream_projects,742,100.000000
1,uses_dependabot_ever,580,78.167116
2,no_dependabot_evidence_ever,162,21.832884
3,has_dependabot_pr,545,73.450135
4,has_dependabot_config,209,28.167116
5,has_both_pr_and_config,174,23.450135
6,pr_only,371,50.000000
7,config_only,35,4.716981
8,security_then_version,147,19.811321


Repo-level lifecycle summary


,dependabot_lifecycle,downstream_projects,percent_of_downstream_projects
2,security_only_ever,371,50.000000
0,never_detected,162,21.832884
3,security_then_version,147,19.811321
4,version_config_only_ever,35,4.716981
1,same_day_security_and_version,21,2.830189
5,version_then_security,6,0.808625


Repo-level usage type summary


,dependabot_usage_type_ever,downstream_projects,percent_of_downstream_projects
2,security_updates_only,371,50.000000
0,both_security_and_version_updates,174,23.450135
1,not_using_dependabot,162,21.832884
3,version_updates_config_only,35,4.716981


Pair-level adoption-time summary


,dependabot_status_at_adoption,pairwise_rows,downstream_projects,cves,percent_of_pairwise_rows
0,no_dependabot_evidence_at_adoption,993,518,168,59.212880
1,security_only_at_adoption,542,251,133,32.319618
2,version_updates_enabled_at_adoption,142,74,44,8.467501


## Sanity Checks

In [7]:
assert len(pairwise) == len(pairs)
assert repo_classified['repo'].nunique() == pairs['normalized_downstream_repo'].nunique()
assert pairwise['dependabot_status_at_adoption'].notna().all()
assert pairwise['uses_dependabot_at_adoption'].notna().all()
assert pairwise['any_dependabot_at_adoption'].notna().all()
assert pairwise['dependabot_setting_at_adoption'].notna().all()
assert pairwise['adoption_time_setting_precise'].notna().all()
assert pairwise['include_in_precise_setting_delay_analysis'].notna().all()
assert pairwise['repo_strategy_security_then_version'].notna().all()

print('Sanity checks passed.')
print(f'Pairwise rows classified: {len(pairwise):,}')
print(f'Downstream projects classified: {repo_classified["repo"].nunique():,}')

Sanity checks passed.
Pairwise rows classified: 1,677
Downstream projects classified: 742
